In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from config import CONFIG_PATH, configure_self_training, load_config
from fifteen_puzzle import PuzzleWalkDataset

config = configure_self_training(load_config())
random_dataset_path = Path(config["dataset"]["save_path"])
if not random_dataset_path.is_absolute():
    random_dataset_path = CONFIG_PATH.parent / random_dataset_path

if random_dataset_path.exists():
    with np.load(random_dataset_path) as saved:
        random_dataset = PuzzleWalkDataset(
            saved["states"], saved["moves"], saved["path_ids"], saved["steps"]
        )
else:
    random_dataset = PuzzleWalkDataset.generate(
        config["walk"]["paths"], config["walk"]["T"], config["seed"]
    )
    random_dataset.save(random_dataset_path)

temperatures = (1.0, 0.5)
self_training_temperature = config["self_training"]["temperature"]
model_specs = [
    (f"Model walk (temperature {temperature})", "model", temperature)
    for temperature in temperatures
]
model_specs.extend(
    (
        f"Model {number} walk (temperature {self_training_temperature})",
        f"model{number}",
        self_training_temperature,
    )
    for number in range(2, config["self_training"]["iterations"] + 2)
)


In [ ]:
from collections import deque
import contextlib
import gc
import io

import torch

from solve import load_model


def walking_distance_state(state, axis):
    counts = [0] * 16
    for position, tile in enumerate(state):
        if tile:
            current_line = position // 4 if axis == 0 else position % 4
            goal_line = (int(tile) - 1) // 4 if axis == 0 else (int(tile) - 1) % 4
            counts[current_line * 4 + goal_line] += 1
    blank_position = int(np.flatnonzero(np.asarray(state) == 0)[0])
    blank_line = blank_position // 4 if axis == 0 else blank_position % 4
    return blank_line, tuple(counts)


def walking_distance_table(axis):
    goal = np.arange(1, 17).reshape(4, 4)
    goal[3, 3] = 0
    start = walking_distance_state(goal.reshape(-1), axis)
    distances = {start: 0}
    queue = deque([start])

    while queue:
        blank_line, counts = queue.popleft()
        for next_line in (blank_line - 1, blank_line + 1):
            if 0 <= next_line < 4:
                for goal_line in range(4):
                    tile_index = next_line * 4 + goal_line
                    if counts[tile_index]:
                        next_counts = list(counts)
                        next_counts[blank_line * 4 + goal_line] += 1
                        next_counts[tile_index] -= 1
                        next_state = next_line, tuple(next_counts)
                        if next_state not in distances:
                            distances[next_state] = distances[(blank_line, counts)] + 1
                            queue.append(next_state)
    return distances


row_distances = walking_distance_table(0)
column_distances = walking_distance_table(1)


def walking_distance(state):
    return row_distances[walking_distance_state(state, 0)] + column_distances[
        walking_distance_state(state, 1)
    ]


def model_dataset_path(model_key, temperature):
    if model_key == "model":
        path = config["dataset"]["model_sampled_save_paths"][str(float(temperature))]
    else:
        path = config["dataset"][f"{model_key}_sampled_save_path"]
    path = Path(path)
    return path if path.is_absolute() else CONFIG_PATH.parent / path


def load_or_generate_distances(model_key, temperature, name):
    dataset_path = model_dataset_path(model_key, temperature)
    walk_length = config["walk"]["T"]

    if dataset_path.exists():
        with np.load(dataset_path) as saved:
            final_states = saved["states"][saved["steps"] == walk_length]
        print(f"Loaded dataset: {dataset_path}")
        return np.fromiter(
            (walking_distance(state) for state in final_states),
            dtype=int,
            count=len(final_states),
        )

    model = load_model(config, model_key=model_key)
    paths = config["walk"]["paths"]
    state_count = paths * walk_length
    states = np.empty((state_count, 16), dtype=np.uint8)
    moves = np.empty(state_count, dtype=np.int64)
    final_distances = np.empty(paths, dtype=int)
    batch_paths = 256
    device = next(model.parameters()).device

    for start in range(0, paths, batch_paths):
        batch_count = min(batch_paths, paths - start)
        with contextlib.redirect_stdout(io.StringIO()):
            batch = PuzzleWalkDataset.generate_from_model(
                model,
                paths=batch_count,
                length=walk_length,
                seed=config["seed"] + start,
                temperature=temperature,
            )
        state_start = start * walk_length
        state_stop = state_start + len(batch)
        states[state_start:state_stop] = batch.states
        moves[state_start:state_stop] = batch.moves
        batch_final_states = batch.states[walk_length - 1 :: walk_length]
        final_distances[start : start + batch_count] = [
            walking_distance(state) for state in batch_final_states
        ]
        print(
            f"Generating {name}: {start + batch_count}/{paths} "
            f"({(start + batch_count) * 100 // paths}%)",
            end=chr(13),
            flush=True,
        )
        del batch
        if device.type == "mps":
            torch.mps.empty_cache()

    print()
    path_ids = np.repeat(np.arange(paths), walk_length)
    steps = np.tile(np.arange(1, walk_length + 1), paths)
    dataset = PuzzleWalkDataset(states, moves, path_ids, steps)
    dataset.save(dataset_path)
    print(f"Saved dataset: {dataset_path}")
    del dataset, states, moves, path_ids, steps, model
    gc.collect()
    if device.type == "mps":
        torch.mps.empty_cache()
    return final_distances


walk_length = config["walk"]["T"]
random_final_states = random_dataset.states[random_dataset.steps == walk_length]
final_distances = {
    "Random walk": np.fromiter(
        (walking_distance(state) for state in random_final_states),
        dtype=int,
        count=len(random_final_states),
    )
}

def is_solvable(state):
    tiles = state[state != 0]
    inversions = sum(
        tiles[left] > tiles[right]
        for left in range(len(tiles))
        for right in range(left + 1, len(tiles))
    )
    blank_row_from_bottom = 4 - int(np.flatnonzero(state == 0)[0]) // 4
    return (inversions + blank_row_from_bottom) % 2 == 1


def sample_solvable_states(count, seed, target_parity):
    rng = np.random.default_rng(seed)
    states = np.empty((count, 16), dtype=np.uint8)
    distances = np.empty(count, dtype=int)
    index = 0

    while index < count:
        state = rng.permutation(16)
        if not is_solvable(state):
            one = int(np.flatnonzero(state == 1)[0])
            two = int(np.flatnonzero(state == 2)[0])
            state[one], state[two] = state[two], state[one]
        distance = walking_distance(state)
        if distance % 2 == target_parity:
            states[index] = state
            distances[index] = distance
            index += 1
    return states, distances


sample_count = config["walk"]["paths"]
independent_random_states, independent_random_distances = sample_solvable_states(
    sample_count, config["seed"] + 1, walk_length % 2
)

for index, (name, model_key, temperature) in enumerate(model_specs):
    final_distances[name] = load_or_generate_distances(
        model_key, temperature, name
    )
    if index == 0:
        final_distances["Independent random states"] = independent_random_distances

maximum_distance = int(max(values.max() for values in final_distances.values()))
bin_edges = np.arange(-0.5, maximum_distance + 1.5, 1)
distance_values = bin_edges[:-1] + 0.5
distributions = {
    name: np.histogram(values, bins=bin_edges)[0]
    for name, values in final_distances.items()
}
maximum_count = max(counts.max() for counts in distributions.values())


In [ ]:
colors = {"Random walk": "steelblue"}
colors.update(
    {
        name: f"C{index % 10}"
        for index, name in enumerate(
            (name for name in final_distances if name != "Random walk")
        )
    }
)

for name, counts in distributions.items():
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(distance_values, counts, width=0.85, color=colors[name])
    if name == "Independent random states":
        ax.set_title("Independent random states walking-distance estimates")
        ax.set_ylabel("Number of random states")
    else:
        ax.set_title(f"{name} final-state walking-distance estimates")
        ax.set_ylabel("Number of paths")
    ax.set_xlabel("Walking-distance estimate from solved state")
    ax.set_xlim(bin_edges[0], bin_edges[-1])
    ax.set_ylim(0, maximum_count * 1.05)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    plt.show()
